In [ ]:
import pandas as pd

# 1. Load the CSV dataset
CSV_DATA_PATH = r"C:\Project\data\flower_prices_with_festivals_2010_2016.csv"
df_raw = pd.read_csv(CSV_DATA_PATH)

# 2. Convert DATE column to datetime format
df_raw['DATE'] = pd.to_datetime(df_raw['DATE'])

# 3. Determine actual date range boundaries
min_date = df_raw['DATE'].min()
max_date = df_raw['DATE'].max()

# 4. Generate expected contiguous daily range
expected_dates = pd.date_range(start=min_date, end=max_date, freq='D')

# 5. Check missing dates and count unique calendar days
missing_dates = expected_dates.difference(df_raw['DATE'])
unique_dates_count = df_raw['DATE'].nunique()
total_rows = len(df_raw)

print("--- STEP 1: TEMPORAL CONTINUITY REPORT (CSV) ---")
print(f"Date Range: {min_date.date()} to {max_date.date()}")
print(f"Total Rows in CSV: {total_rows}")
print(f"Unique Calendar Days: {unique_dates_count}")
print(f"Expected Daily Count: {len(expected_dates)}")
print(f"Missing Calendar Dates: {len(missing_dates)}")

if len(missing_dates) == 0:
    print("\n[VERIFIED SUCCESS]: 100% complete temporal continuity (0 missing dates).")
else:
    print(f"\n[WARNING]: Detected {len(missing_dates)} missing dates.")
    print("Missing dates sample:", missing_dates[:5].strftime('%Y-%m-%d').tolist())

if total_rows > unique_dates_count:
    print(f"\n[NOTE]: Found {total_rows - unique_dates_count} duplicate date rows due to same-day festival overlaps.")
    duplicate_rows = df_raw[df_raw.duplicated(subset=['DATE'], keep=False)].sort_values('DATE')
    
    # 2. Group by DATE to aggregate co-occurring festivals into a clean view
    dup_summary = duplicate_rows.groupby('DATE').agg({
        'festival_name': lambda x: ' | '.join(x.dropna().astype(str)),
        'price': 'first'
    }).reset_index()

    print("\n--- DUPLICATE DATES SUMMARY (Co-occurring Festivals) ---")
    print(dup_summary.to_string(index=False))

--- STEP 1: TEMPORAL CONTINUITY REPORT (CSV) ---
Date Range: 2010-01-01 to 2016-12-31
Total Rows in CSV: 2581
Unique Calendar Days: 2557
Expected Daily Count: 2557
Missing Calendar Dates: 0

[VERIFIED SUCCESS]: 100% complete temporal continuity (0 missing dates).

[NOTE]: Found 24 duplicate date rows due to same-day festival overlaps.

--- DUPLICATE DATES SUMMARY (Co-occurring Festivals) ---
      DATE                       festival_name  price
2010-09-22 Ganesh Visarjan | Anant Chaturdashi    405
2010-10-17            Vijayadashami | Dussehra    405
2010-11-05               Lakshmi Puja | Diwali    405
2011-09-11 Anant Chaturdashi | Ganesh Visarjan    230
2011-10-04      Saraswati Puja | Durga Ashtami    820
2011-10-06            Vijayadashami | Dussehra    820
2011-10-26               Diwali | Lakshmi Puja    520
2012-09-29 Anant Chaturdashi | Ganesh Visarjan    610
2012-10-24            Dussehra | Vijayadashami    820
2012-11-13               Lakshmi Puja | Diwali    310
2013-09-18 